## GRU vs LSTM
- gate 2개  필요하면 기억유지 아니면 갱신
    - Update gate
    - Reset gate

- gate 3개  : 단기기억 + 장기기억, 분리관리
    - Forget gate
    - Input gate
    - Output gate
    - state 관리
        - hidden state
        - cell state
- 성능
    - 실제로 차이가 많이나는 경우가 드묾
- 사용은
    - GRU
        - 데이터 적음, GPU 약할, 빠른결과, 실시간 추론
        - 감성분석, 간단한 쳇봇, Iot
    - LSTM
        - 긴 문맥, 복잡한 시계역, 장기의존성 중요
        - 음성인식, 번역, 금융시계열

- Baseline으로서 GRU를 이용해서 성능을 빠르게 확인
- 빠르게 GRU, 긴 문맥 LSTM, 최고성능... Transformer

## GRU MODEL

In [ ]:
import torch.nn as nn

class GRUClassification(nn.Module):
    def __init__(self, vocab_size, embed_dim=128,hidden_dim = 64):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.gru = nn.GRU(
            input_size= embed_dim,
            hidden_size = hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear( hidden_dim , 1)
    def forward(self, x):
        x = self.embedding(x)
        output, hidden =self.gru(x)
        # hidden shape  (num_layer, batch, hidden_dim)   (1, batch , 64)        
        x = self.fc()  # 2 dim
        return x

## Data

In [2]:
import  torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

In [8]:
# url = 'https://github.com/skn29teacher/skn29_lecture/blob/main/data_nlp/daum_movie_review.csv'
import pandas as pd
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
sample_df = df.iloc[-1000:]

In [ ]:
sample_df['target'] = sample_df['rating'].apply(lambda x : 1 if x>0.5 else 0)

## 한글만 남기기

In [ ]:

import re
def clean_text(text):
    return re.sub(r'[^가-힣\s]', '', text).strip()
df['clean'] = df['review'].apply(lambda x : clean_text( x ) )

## 토큰화 : 숫자로변경하기위한 최소한의 단어들

In [ ]:
#from konlpy.tag import Okt
# 공백을기준으로 분류
df['tokens'] = df['clean'].apply(lambda x : x.split() )

## 단어사전 구축하기

In [ ]:
counter = Counter()
for tokens in df['tokens']:
    counter.update(tokens)
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1,
}    
for word, freq in counter.items():
    if len(word)>=2:
        vocab[word] = len(vocab)
print(f'단어수 : {len(vocab)}')

## 문장을 숫자로 & 패딩
